0.1 加载模型DeepSeek-R1-Distill-Qwen-14B 并初次测试

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# 指定模型目录
model_dir = "/home/wangshuo/resource/AIModels/NLP/deepseek/DeepSeek-R1-Distill-Qwen-14B"

# 加载分词器和模型
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForCausalLM.from_pretrained(model_dir)

# 创建文本生成pipeline
text_generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)  # device=0表示使用第一个GPU

# 输入提示
input_prompt = "Who are you?"

# 生成文本
output = text_generator(input_prompt, max_length=50, num_return_sequences=1)

# 输出结果
print(output[0]['generated_text'])


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 100.00 MiB. GPU 0 has a total capacity of 23.68 GiB of which 19.81 MiB is free. Including non-PyTorch memory, this process has 23.66 GiB memory in use. Of the allocated memory 23.41 GiB is allocated by PyTorch, and 678.00 KiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

0.2 加载DeepSeek-R1-Distill-Qwen-14B模型

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 指定模型目录
model_dir = "/home/wangshuo/resource/AIModels/NLP/deepseek/DeepSeek-R1-Distill-Qwen-14B"

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
tokenizer.padding_side = 'left'

# 设备优化
torch.backends.cuda.matmul.allow_tf32 = True  # 允许 TF32 加速

# 加载模型，并使用 device_map 参数进行多GPU分配
model = AutoModelForCausalLM.from_pretrained(model_dir, device_map="auto", trust_remote_code=True)

# 将模型设置为评估模式
model.eval()  # 半精度加速
model = torch.compile(model)  # 进一步优化计算图



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

0.3 测试打标签的能力

In [ ]:
import  re
# **正则匹配函数**
def match_answer(final_output):
    match = re.search(r"</think>\s*\n*(?i)(yes|no)\s*,\s*([0-9]*\.[0-9]+)", final_output)
    if match:
        return {"oracle_label": match.group(1).lower(), "oracle_confidence_probability": float(match.group(2))}
    return {"oracle_label": None, "oracle_confidence_probability": None}

In [3]:
# input_text = "Trump now 180k in FL with nothing but red votes left to count. He is going to massively outperform 2016"  # 替换为您的输入文本
# input_text = "Trump has been holding steady in Pennsylvania with 110,000 vote lead for a while now. I also hear that some red areas will be coming in. Hopefully he can hold on and take that state and then we can beat back the cheat in Georgia."  # 替换为您的输入文本
# input_text = "FIGHT FOR TRUMP - SAVE AMERICA - SAVE THE WORLD "

# input_text = "Trump Is a machine. One rally after another, day after day and he never looks tired.  Hillary was a much better candidate than Biden and the rationale for electing Trump now is greater than it was in 2016, because now we know he can deliver on all those promises.  Simple logic dictates, Trump wins."
# input_text = "Trump will be President; charismatic Christian leaders say that the election is not over, and say that Trump will be re-elected  ECHO ECHO ECHO!!!!! Make this go viral!  Stand With President Trump  #Trump2020LandslideVictory #Trump2020Landslide #Trump2020 #TRUMP2020ToSaveAmerica #AmericaFirst #VoteRed2020 #VoteRedToSaveAmerica2020 #VoteRedToSaveAmerica #RedWave #RedWave2020 #WeThePeople #1A #2A #2AShallNOTBeInfringed #NRA #USA #PJNET #AllLivesMatter #StandUpForAmerica #TrumpRally #TrumpRally2020 #SCOTUS #SupremeCourt #PromisesMadePromisesKept #PromisesKept #stopthesteal #StopTheCoup #4MoreYears #FourMoreYears #AuditTheVote #AuditTheElection #AuditTheBallots #LegalVotesOnly #CountEveryLegalVote #DeclassifyEverything #DominionVotingSystems #GodWins #HoldTheLine #echo #viral #Trending #Trend"
# input_text = "Trump Campaign Senior Advisor Stephen Miller: We have more than enough time to right the wrong of this fraudulent election result"
# input_text = "fight for trump - save america - save the world "
# input_text = "Flashback: Former Obama Vice President JOE BIDEN Was Accused of CLASS A FELONY CHARGES In UkraineECHO ECHO ECHO!!!!!! Make this go viral!Stand With President Trump"

# input_text =',Barr Appoints Durham Special Counsel for Investigation Into Origins of Russia Probe - WSJECHO ECHO ECHO!!!!!!  Make this go viral! Stand With President Trump#Trump2020LandslideVictory #Trump2020Landslide #Trump2020 #TRUMP2020ToSaveAmerica #AmericaFirst #VoteRed2020 #VoteRedToSaveAmerica2020 #VoteRedToSaveAmerica #RedWave #RedWave2020 #WeThePeople #1A #2A #2AShallNOTBeInfringed #NRA #USA #PJNET #AllLivesMatter #StandUpForAmerica #TrumpRally #TrumpRally2020 #SCOTUS #SupremeCourt #PromisesMadePromisesKept #PromisesKept #stopthesteal #StopTheCoup #4MoreYears #FourMoreYears #AuditTheVote #AuditTheVotes #AuditTheElection  #AuditTheBallots #LegalVotesOnly #CountEveryLegalVote #OnlyLegalVotes #DeclassifyEverything #DominionVotingSystems #GodWins #HoldTheLine #ILoveAmerica #DoNotConcede #SwampPanic #FightBack #BidenCheated #BidenCheats #echo #viral #Trending #Trend'
# input_text = "In case you missed it:"
input_text = "America needs its captain more than ever"
topic = "Trump will win and save America"

# 构建提示
input_prompt = f"Text: {input_text}.\n\n Given the topic '{topic}', does the above text imply or infer this topic? Answer 'yes' or 'no'.followed by a comma and a confidence probability (0 - 1).no explanations.\n"

# 编码输入
inputs = tokenizer(input_prompt, return_tensors="pt")

# 将输入移动到模型所在的设备
inputs = {key: value.to(model.device) for key, value in inputs.items()}
# 生成文本
with torch.no_grad():
    outputs = model.generate(**inputs, max_length=512,do_sample=False,temperature=0.0,
        top_p=0.0,pad_token_id=tokenizer.eos_token_id  # 解决填充问题
        )
# 解码输出
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
# 输出结果
print(generated_text)

# # 获取输入提示的 token 数量
# prompt_length = inputs['input_ids'].shape[1]
# 
# # 从生成的输出中切除提示部分，只保留模型生成的 tokens
# generated_tokens = outputs[0][prompt_length:]
# final_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)
# print(final_output)
# print(match_answer(final_output))

Text: America needs its captain more than ever.

 Given the topic 'Trump will win and save America', does the above text imply or infer this topic? Answer 'yes' or 'no'.followed by a comma and a confidence probability (0 - 1).no explanations, no <think> label.
Okay, so I need to figure out if the text "America needs its captain more than ever." implies or infers the topic "Trump will win and save America." Hmm, let's break this down.

First, the text is a statement about America needing its captain. The word "captain" here is likely a metaphor, referring to a leader, probably a president. So, it's saying that America requires strong leadership now more than ever.

Now, the topic is about Trump winning and saving America. So, does the original text directly talk about Trump? No, it doesn't mention him by name. But it's about needing a leader, which could be interpreted as needing Trump if the context is about him running again.

But wait, the text doesn't specify who the captain is. It 

0.4 测试： 获取输入的token数量

In [ ]:
import re

# 获取输入提示的 token 数量
prompt_length = inputs['input_ids'].shape[1]
print(prompt_length)
# 从生成的输出中切除提示部分，只保留模型生成的 tokens
generated_tokens = outputs[0][prompt_length:]
final_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)
print(final_output)

# oracle_dict = match_answer(final_output)
# 输出结果
# print(oracle_dict)


1.1 使用DeepSeek-R1-Distill-Qwen-14B 进行推理 V1 版代码

In [ ]:
import torch
import pandas as pd
import re
from tqdm import tqdm

data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/valid_data/'
INPUT_CSV = data_dir + "posts_test_pure.csv"      # 输入文件
OUTPUT_CSV = data_dir + "post_result.csv"    # 输出文件
BATCH_SIZE = 8                    # 批处理大小
TOPIC = "Trump will win and save America"  # 主题

# **正则匹配函数**
def match_answer(final_output):
    match = re.search(r"</think>\s*\n*(?i)(yes|no)\s*,\s*([0-9]*\.[0-9]+)", final_output)
    if match:
        return {"oracle_label": match.group(1).lower(), "oracle_confidence_probability": float(match.group(2))}
    return {"oracle_label": None, "oracle_confidence_probability": None}
# **读取 CSV**
df = pd.read_csv(INPUT_CSV)
if "body" not in df.columns:
    raise ValueError("CSV 文件缺少 'body' 列")

# **构造输入文本**
prompts = [
    f"Given the topic '{TOPIC}', does the following text imply this topic? Answer 'yes' or 'no'. followed by a comma and a confidence probability (a float between 0 and 1). No explanation is needed.Don't think too much, give answers quickly and accurately.\n\nText: {text}\n"
    for text in df["body"].astype(str)
]

# **批量处理**
results = []
for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc="Processing"):
    batch_prompts = prompts[i:i + BATCH_SIZE]

    # **编码输入**
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    # **模型推理**
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=1024, do_sample=False, temperature=0.0, top_p=0.0)

    # **解码输出**
    for j, output in enumerate(outputs):
        prompt_length = inputs['input_ids'][j].shape[0]  # 获取当前输入的 token 数量
        generated_text = tokenizer.decode(output, skip_special_tokens=True)
        print(generated_text)
        generated_tokens = output[prompt_length:]  # 切除输入部分
        final_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        results.append(match_answer(final_output))

# **存储结果**
df["oracle_label"] = [r["oracle_label"] for r in results]
df["oracle_confidence_probability"] = [r["oracle_confidence_probability"] for r in results]
df.to_csv(OUTPUT_CSV, index=False)

print(f"处理完成，结果已保存到 {OUTPUT_CSV}")

1.2 使用DeepSeek-R1-Distill-Qwen-14B 进行推理 V2 版代码

In [ ]:
import os
import torch
import pandas as pd
import re
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

# 配置文件路径和批处理参数
data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/valid_data/'
INPUT_CSV = data_dir + "posts_test.csv"      # 输入文件
OUTPUT_CSV = data_dir + "post_result.csv"    # 输出文件
BATCH_SIZE = 8                              # 批处理大小
TOPIC = "Trump will win and save America"   # 主题

# 正则匹配函数，用于从模型输出中提取答案和置信度
def match_answer(final_output):
    match = re.search(r"</think>\s*\n*(?i)(yes|no)\s*,\s*([0-9]*\.[0-9]+)", final_output)
    if match:
        return {"oracle_label": match.group(1).lower(), "oracle_confidence_probability": float(match.group(2))}
    return {"oracle_label": None, "oracle_confidence_probability": None}

# 读取 CSV 文件，要求必须包含 'body' 列
df = pd.read_csv(INPUT_CSV)
if "body" not in df.columns:
    raise ValueError("CSV 文件缺少 'body' 列")

# 构造每一行的提示文本，将 'body' 列作为输入文本
prompts = [
    f"Given the topic '{TOPIC}', does the following text imply this topic? Answer 'yes' or 'no'. followed by a comma and a confidence probability (a float between 0 and 1). No explanation is needed.\n\nText: {text}\nAnswer:"
    for text in df["body"].astype(str)
]

# 如果输出文件已存在，先删除以避免追加旧数据
if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)

# 加载分词器和模型
MODEL_DIR = "/home/wangshuo/resource/AIModels/NLP/deepseek/DeepSeek-R1-Distill-Qwen-14B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, device_map="auto", trust_remote_code=True)
model.eval()

# 批量处理：逐批推理并写入结果
for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc="Processing"):
    batch_prompts = prompts[i:i + BATCH_SIZE]
    
    # 编码当前批次的提示文本（padding、truncation 保证所有输入长度一致）
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    
    # 模型推理，设置最大生成 token 数量（这里 max_length 包含输入和生成部分）
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=500, do_sample=False, temperature=0.0, top_p=0.0)
    
    batch_results = []
    # 对当前批次的每一条生成结果进行解码和后处理
    for j, output in enumerate(outputs):
        prompt_length = inputs['input_ids'][j].shape[0]  # 当前输入的 token 数量
        generated_tokens = output[prompt_length:]          # 切除输入部分，保留模型生成的 tokens
        final_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        batch_results.append(match_answer(final_output))
    
    # 将当前批次的结果添加到原始数据中（取对应的行）
    batch_df = df.iloc[i:i + BATCH_SIZE].copy()
    batch_df["oracle_label"] = [r["oracle_label"] for r in batch_results]
    batch_df["oracle_confidence_probability"] = [r["oracle_confidence_probability"] for r in batch_results]
    
    # 将当前批次的 DataFrame 追加写入 CSV 文件
    # 如果是第一个批次则写入头部，否则追加写入（header=False）
    if i == 0:
        batch_df.to_csv(OUTPUT_CSV, index=False, mode='w', header=True)
    else:
        batch_df.to_csv(OUTPUT_CSV, index=False, mode='a', header=False)

print(f"处理完成，结果已保存到 {OUTPUT_CSV}")


1.3 使用DeepSeek-R1-Distill-Qwen-14B 进行推理 V3 版代码

In [ ]:

# 正则匹配函数，用于从模型输出中提取答案和置信度
def match_answer_ex(final_output):
    # 提取 </think> 后的文本
    think_split = final_output.split("</think>", 1)
    if len(think_split) > 1:
        answer_text = think_split[1].strip()
    else:
        answer_text = None

    # 使用正则表达式提取 'yes' 或 'no' 以及置信度
    match = re.search(r"(?i)\b(yes|no)\b\s*,\s*([0-9]*\.[0-9]+)", answer_text or "")
    if match:
        oracle_label = match.group(1).lower()
        oracle_confidence_probability = float(match.group(2))
    else:
        oracle_label = None
        oracle_confidence_probability = None

    return {
        "oracle_label": oracle_label,
        "oracle_confidence_probability": oracle_confidence_probability,
        "answer": answer_text
    }



# 批量处理：逐批推理并写入结果
for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc="Processing"):
    batch_prompts = prompts[i:i + BATCH_SIZE]
    
    # 编码当前批次的提示文本（padding、truncation 保证所有输入长度一致）
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    
    # 模型推理，设置最大生成 token 数量（这里 max_length 包含输入和生成部分）
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=500, do_sample=False, temperature=0.0, top_p=0.0)
    
    batch_results = []
    # 对当前批次的每一条生成结果进行解码和后处理
    for j, output in enumerate(outputs):
        prompt_length = inputs['input_ids'][j].shape[0]  # 当前输入的 token 数量
        generated_tokens = output[prompt_length:]          # 切除输入部分，保留模型生成的 tokens
        final_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        batch_results.append(match_answer_ex(final_output))
    
    # 将当前批次的结果添加到原始数据中（取对应的行）
    batch_df = df.iloc[i:i + BATCH_SIZE].copy()
    batch_df["oracle_label"] = [r["oracle_label"] for r in batch_results]
    batch_df["oracle_confidence_probability"] = [r["oracle_confidence_probability"] for r in batch_results]
    batch_df["answer"] = [r["answer"] for r in batch_results]
    
    # 将当前批次的 DataFrame 追加写入 CSV 文件
    # 如果是第一个批次则写入头部，否则追加写入（header=False）
    if i == 0:
        batch_df.to_csv(OUTPUT_CSV, index=False, mode='w', header=True)
    else:
        batch_df.to_csv(OUTPUT_CSV, index=False, mode='a', header=False)

print(f"处理完成，结果已保存到 {OUTPUT_CSV}")


0.5 数据预处理：读取posts_valid.csv中的数据，将body字段中#trump2020tosaveamerica ，#+word的词全部删除，如果删除后posts的body字段为空，则删除该post，将处理后的post数据保存到posts_pure.csv中

In [ ]:
import pandas as pd
import re

data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/valid_data/'
# 读取CSV文件
df = pd.read_csv(data_dir + "posts_test.csv")

def remove_hashtags(text):
    if not isinstance(text, str):
        return text
    # 使用正则删除所有以#开头的连续字母数字下划线组合（即匹配#trump2020tosaveamerica以及其它类似#word的词）
    cleaned = re.sub(r"#\w+", "", text)
    # 清理多余的空格
    return cleaned.strip()

# 对body字段进行处理
df["body"] = df["body"].apply(remove_hashtags)

# 删除body为空的行（如果经过清洗后为空字符串或仅包含空白字符）
df_pure = df[df["body"].str.strip() != ""]

# 保存处理后的数据到新的CSV文件
df_pure.to_csv(data_dir + "posts_test_pure.csv", index=False)


1.4 使用DeepSeek-R1-Distill-Qwen-14B 进行推理 V4 版代码

步骤1：

In [2]:
import os
import torch
import pandas as pd
import re
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

# data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/valid_data/'
# data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
data_dir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'

# 定义答案解析函数，匹配答案行格式（严格要求： "yes,0.85" 或 "no,0.85"）
def match_answer_ex(answer_line):
    # 清理前后空格，并剔除前缀 "Answer:"（若存在）
    cleaned = answer_line.strip()
    if cleaned.lower().startswith("answer:"):
        cleaned = cleaned[len("answer:"):].strip()
    print(cleaned)
    # 使用严格的正则表达式匹配格式
    pattern = r"^(?i)(yes|no)\s*,\s*([0-9]*\.[0-9]+)$"
    match = re.search(pattern, cleaned)
    if match:
        return {
            "oracle_label": match.group(1).lower(),
            "oracle_confidence_probability": float(match.group(2)),
            "answer": cleaned
        }
    return {"oracle_label": None, "oracle_confidence_probability": None, "answer": cleaned}


# 加载分词器和模型
MODEL_DIR = "/home/wangshuo/resource/AIModels/NLP/deepseek/DeepSeek-R1-Distill-Qwen-14B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
tokenizer.padding_side = 'left'  # 关键修改

# 设备优化
torch.backends.cuda.matmul.allow_tf32 = True  # 允许 TF32 加速
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, device_map="auto", trust_remote_code=True)
model.half()
model.eval()  
model = torch.compile(model)  # 进一步优化计算图

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

步骤2：

In [3]:
INPUT_CSV = data_dir + "post_ML1_albert_mnli.csv"      # 输入文件
OUTPUT_CSV = data_dir + "post_LLM.csv"    # 输出文件
# TOPIC = "Trump will win and save America"   # 主题
TOPIC = "I support Trump"   # 主题
BATCH_SIZE = 32                              # 批处理大小
START_ROW = 1  # 指定起始行号（不包括 header），例如：从第1000行开始读取
N = 30000         # 读取 N 行数据

# 读取 CSV 文件，仅从指定行开始读取 N 行，保留表头
# 注意：header 默认在第一行 (index 0)，因此 skiprows 从 1 开始跳过
df = pd.read_csv(INPUT_CSV, skiprows=range(1, START_ROW), nrows=N)
if "body" not in df.columns:
    raise ValueError("CSV 文件缺少 'body' 列")

# 构造每一行的提示文本，将 'body' 列作为输入文本
prompts = [
    f"Text: {text}.\n\n Given the topic '{TOPIC}', does the following text imply or infer this topic? Only answer 'yes' or 'no'. No explanation just the answer.\n"
    for text in df["body"].astype(str)
]

# 如果输出文件已存在，先删除以避免追加旧数据
if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)

步骤3：

In [4]:
# 批量处理：逐批推理并写入结果
results = []  # 存储所有结果
for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc="Processing"):
    batch_prompts = prompts[i:i + BATCH_SIZE]
    
    # 编码当前批次的提示文本
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    
    # 模型推理
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=500,do_sample=False,temperature=0.0,
        top_p=0.0,)
    
    batch_results = []
    # 对当前批次的每一条生成结果进行解码和后处理
    for j, output in enumerate(outputs):
        # 解码完整生成的文本
        full_text = tokenizer.decode(output, skip_special_tokens=True)
        # 分割文本为多行，并取最后一行（答案所在行）
        lines = full_text.strip().splitlines()
        if lines:
            answer_line = lines[-1].strip()
        else:
            answer_line = ""
        # 使用 match_answer_ex 函数解析答案行
        batch_results.append(match_answer_ex(answer_line))
    
    # 将当前批次的结果添加到对应的 DataFrame 行中
    batch_df = df.iloc[i:i + BATCH_SIZE].copy()
    batch_df["oracle_label"] = [r["oracle_label"] for r in batch_results]
    # batch_df["oracle_confidence_probability"] = [r["oracle_confidence_probability"] for r in batch_results]
    batch_df["answer"] = [r["answer"] for r in batch_results]
    
    # 追加写入 CSV 文件：第一个批次写入表头，其余批次追加写入
    if i == 0:
        batch_df.to_csv(OUTPUT_CSV, index=False, mode='w', header=True)
    else:
        batch_df.to_csv(OUTPUT_CSV, index=False, mode='a', header=False)
    
    results.extend(batch_results)

print(f"处理完成，结果已保存到 {OUTPUT_CSV}")


Processing:   0%|          | 0/938 [00:00<?, ?it/s]/home/wangshuo/software/anaconda3/envs/iogs/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/wangshuo/software/anaconda3/envs/iogs/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
/tmp/ipykernel_690437/3114043334.py:21: DeprecationWarning: Flags not at the start of the expression '^(?i)(yes|no)\\s*,\\s*' (truncated)
  match = re.search(pattern, cleaned)
Processing:   0%|          | 1/938

no
no
yes
no
No.
yes
no
no
no
no
yes
yes
No.
no
no
So, after considering all this, I think the text doesn't explicitly or strongly imply support for Trump. The political elements are
no
yes
no
no
no
yes
no
Wait, but the user is using hashtags or mentions related to MAGA, which is Trump's movement. So, maybe the inference is that they support Trump by aligning with MAGA. But the original text doesn
no
yes
no
yes
yes
yes
yes
yes


Processing:   0%|          | 2/938 [02:22<18:34:10, 71.42s/it]/home/wangshuo/software/anaconda3/envs/iogs/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/wangshuo/software/anaconda3/envs/iogs/lib/python3.8/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


no
I'm a bit confused. On one hand, the text is general, but the topic is
No.
yes
yes
no
no
yes
yes
yes
yes
no
no
yes
no
no
No
no
no
no
yes
yes
no
yes
no
yes
no
no
yes
yes
no
no


Processing:   0%|          | 2/938 [03:05<24:06:33, 92.73s/it]


KeyboardInterrupt: 

读取post_oracle.csv文件，oracle_label属性为yes的排在前面，no排后，yes内按照oracle_confidence_probability降序排

In [ ]:
import pandas as pd

# Define the file path
data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
input_csv = data_dir + "post_oracle.csv"

# Read the CSV file
df = pd.read_csv(input_csv)

# 将oracle_label列转换为类别类型，并指定排序顺序
df['oracle_label'] = pd.Categorical(df['oracle_label'], categories=['yes', 'no'], ordered=True)

# 按照oracle_label和oracle_confidence_probability进行排序
df_sorted = df.sort_values(by=['oracle_label', 'oracle_confidence_probability'], ascending=[True, False])

# 将排序后的数据保存回CSV文件（可选）
df_sorted.to_csv(data_dir + 'post_oracle_sorted.csv', index=False)

为我的post文件添加oracle_label 和 oracle_confidence_probability属性，对于workload1_label = workload1_entailment 的元组 oracle_label  = yes , oracle_confidence_probability = 0.99, 其他的元组 oracle_label  = no , oracle_confidence_probability = 0.99

In [ ]:
import pandas as pd
data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
# 读取 post 文件（假设文件名为 post.csv）
df = pd.read_csv(data_dir + "posts.csv")

# 根据 workload1_label 添加 oracle_label 和 oracle_confidence_probability 属性
df["oracle_label"] = df["workload1_label"].apply(lambda x: "yes" if x == "workload1_entailment" else "no")
df["oracle_confidence_probability"] = 0.99

# 保存结果到新的 CSV 文件中
df.to_csv(data_dir + "posts.csv", index=False)


posts.csv 对post_oracle_sorted.csv进行关联查询 ，对于post_oracle_sorted文件中oracle_label = yes的元组通过id:ID 查找posts.csv中对应的元组，将posts.csv对应的元组的oracle_label 和oracle_confidence_probability赋值为post_oracle_sorted对应元组的值

In [17]:
import pandas as pd
data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
# 读取两个 CSV 文件
posts_df = pd.read_csv(data_dir + "posts.csv")
oracle_df = pd.read_csv(data_dir + "post_oracle_sorted.csv")

# 筛选 oracle_df 中 oracle_label 为 "yes" 的记录，并保留更新所需的字段（假设关联字段为 "ID"）
oracle_yes_df = oracle_df[oracle_df["oracle_label"] == "deepseek_yes"][["id:ID", "oracle_label", "oracle_confidence_probability"]]

# 通过 ID 关联两个 DataFrame
# 这里采用 merge 方法，将 posts_df 和 oracle_yes_df 左连接（left join），保留 posts_df 中所有记录，
# 如果 oracle_yes_df 中有匹配，则会生成后缀的列
merged_df = posts_df.merge(oracle_yes_df, on="id:ID", how="left", suffixes=("", "_oracle"))

# 对于匹配到的行（即 oracle_yes_df 有记录的行），更新 posts_df 中的 oracle_label 和 oracle_confidence_probability
# 如果 oracle_label_oracle 非空，则说明有匹配记录
merged_df.loc[merged_df["oracle_label_oracle"].notna(), "oracle_label"] = merged_df.loc[merged_df["oracle_label_oracle"].notna(), "oracle_label_oracle"]
merged_df.loc[merged_df["oracle_confidence_probability_oracle"].notna(), "oracle_confidence_probability"] = merged_df.loc[merged_df["oracle_confidence_probability_oracle"].notna(), "oracle_confidence_probability_oracle"]

# 删除 merge 后产生的额外列
merged_df.drop(columns=["oracle_label_oracle", "oracle_confidence_probability_oracle"], inplace=True)

# 保存更新后的结果到新的 CSV 文件
merged_df.to_csv(data_dir + "posts.csv", index=False)

print("关联更新完成，结果已保存到 posts_updated.csv")


关联更新完成，结果已保存到 posts_updated.csv


将data_dir + "post_oracle_sorted.csv" 文件中oracle_label 属性为yes 替换为deepseek_yes

In [18]:
import pandas as pd

data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
input_file = data_dir + "posts.csv"
output_file = data_dir + "posts.csv"

# 读取 CSV 文件
df = pd.read_csv(input_file)

# 将 oracle_label 列中值为 "yes" 的替换为 "deepseek_yes"
df.loc[df["oracle_label"] == "yes", "oracle_label"] = "deepseek_yes"

# 保存修改后的数据到新文件中
df.to_csv(output_file, index=False)

print("处理完成，结果已保存到", output_file)


处理完成，结果已保存到 /home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/posts.csv


[2025-07-03 12:22:32,653] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/wangshuo/software/anaconda3/envs/iogs/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


MissingCUDAException: CUDA_HOME does not exist, unable to compile CUDA op(s)

deepspeed加速-代码未运行成功

In [ ]:
import os
import torch
import pandas as pd
import re
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import deepspeed

# data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_3/valid_data/'
# data_dir = '/home/wangshuo/resource/datasets/IOGS/many_predicates/independent/dataset_4/'
data_dir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'

def match_answer_ex(answer_line):
    cleaned = answer_line.strip()
    if cleaned.lower().startswith("answer:"):
        cleaned = cleaned[len("answer:"):].strip()
    pattern = r"^(?i)(yes|no)\s*,\s*([0-9]*\.[0-9]+)$"
    match = re.search(pattern, cleaned)
    if match:
        return {
            "oracle_label": match.group(1).lower(),
            "oracle_confidence_probability": float(match.group(2)),
            "answer": cleaned
        }
    return {"oracle_label": None, "oracle_confidence_probability": None, "answer": cleaned}

# 1) 加载分词器和模型
MODEL_DIR = "/home/wangshuo/resource/AIModels/NLP/deepseek/DeepSeek-R1-Distill-Qwen-14B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
tokenizer.padding_side = 'left'

# 2) 原生模型加载
raw_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.float16,          # 直接以半精度加载
    device_map="auto",
    trust_remote_code=True
)

# 3) 用 DeepSpeed 初始化推理引擎
engine = deepspeed.init_inference(
    raw_model,
    mp_size=1,                          # 张量并行度，如有多卡可调整
    dtype=torch.float16,                # 半精度推理
    replace_with_kernel_inject=True,    # 自动用 kernel-inject 优化 Transformer 层
    # optional: you 可以在此添加 offload_config 来开启 CPU offload，例：
    # offload_dir=".", offload_config={"device": "cpu", "pin_memory": True}
)

# 4) 保持编译模型：DeepSpeed 会自动做 graph 优化，无需再 torch.compile
# engine = torch.compile(engine)      # 可选

# 推理时直接用 engine.generate
INPUT_CSV = data_dir + "post_ML1_albert_mnli.csv"
OUTPUT_CSV = data_dir + "post_LLM.csv"
TOPIC = "I support Trump"
BATCH_SIZE = 16
START_ROW = 1
N = 30000

df = pd.read_csv(INPUT_CSV, skiprows=range(1, START_ROW), nrows=N)
if "body" not in df.columns:
    raise ValueError("CSV 文件缺少 'body' 列")

prompts = [
    f"Text: {text}.\n\n Given the topic '{TOPIC}', does the following text imply or infer this topic? Only answer 'yes' or 'no'. No explanation just the answer.\n"
    for text in df["body"].astype(str)
]

if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)

results = []
for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc="Processing"):
    batch_prompts = prompts[i:i + BATCH_SIZE]

    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(engine.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = engine.generate(
            **inputs,
            max_new_tokens=600,
            do_sample=False,
            temperature=0.0,
            top_p=0.0,
        )

    batch_results = []
    for output in outputs:
        full_text = tokenizer.decode(output, skip_special_tokens=True)
        lines = full_text.strip().splitlines()
        answer_line = lines[-1].strip() if lines else ""
        batch_results.append(match_answer_ex(answer_line))

    batch_df = df.iloc[i:i + BATCH_SIZE].copy()
    batch_df["oracle_label"] = [r["oracle_label"] for r in batch_results]
    batch_df["answer"] = [r["answer"] for r in batch_results]

    mode = 'w' if i == 0 else 'a'
    header = True if i == 0 else False
    batch_df.to_csv(OUTPUT_CSV, index=False, mode=mode, header=header)

    results.extend(batch_results)

print(f"处理完成，结果已保存到 {OUTPUT_CSV}")
